# 05. 因果推断（JD 加分项）

超越相关性，识别指标波动的因果根因。

涵盖：Granger 因果检验、DoWhy 因果效应估计。

## 1. Granger 因果检验

检验广告投放(ad_spend)是否对销量(sales)有预测作用。

In [ ]:
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
from utils.data_loader import prepare_all
from attribution import GrangerCausality

daily = prepare_all()['daily_sales'].set_index('date')
gc = GrangerCausality(maxlag=7)
print(gc.test(daily, 'sales', 'ad_spend'))
print(gc.test(daily, 'sales', 'is_weekend'))

## 2. 多变量 Granger 因果扫描

In [ ]:
candidates = ['ad_spend', 'is_weekend', 'is_holiday']
result = gc.test_all(daily, 'sales', candidates)
display(result)

## 3. DoWhy 因果效应估计

估计广告投放对销量的平均因果效应（ATE）。

In [ ]:
try:
    from attribution import DoWhyCausalEstimator
    df = daily.reset_index()
    # 二值化处理变量：高投放 vs 低投放
    df['high_ad'] = (df['ad_spend'] > df['ad_spend'].median()).astype(int)
    est = DoWhyCausalEstimator(
        treatment='high_ad',
        outcome='sales',
        common_causes=['is_weekend', 'is_holiday']
    )
    result = est.estimate(df)
    print(result)
except ImportError:
    print('dowhy 未安装，请运行: pip install dowhy')
    print('DoWhy 可估计因果效应并通过安慰剂检验验证稳健性。')

## 小结

- Granger 因果：时序归因的线索，判断变量间预测关系
- DoWhy：基于因果图的效应估计，配合反驳检验确保稳健
- 因果推断是归因分析的「升级版」，从相关走向因果